# ABLATION STUDY

A) No Multi-Agent  (single LLM call, with intent)

B) No Intent       (multi-agent + BERT, no intent)

C) No Legal-BERT   (multi-agent + intent, LLM-only)



## 1. Cài đặt thư viện

In [1]:
!pip install -U transformers peft accelerate langgraph bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 101.6 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: langchain-core
    Found ex

## 2. Import thư viện

In [2]:
import warnings
import logging
 
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
 
import json
import os
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModel, AutoTokenizer
from typing import Dict, Any, Optional, List, Union, TypedDict
import re
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

## 3. Cấu hình HuggingFace Token

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

## 4. Kiểm tra môi trường

In [4]:
print("=== Environment Check ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")
print("=" * 50)

=== Environment Check ===
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 15.64 GB
PyTorch Version: 2.10.0+cu128


## 5. Load Base Model

Tải Mistral-7B-Instruct với cấu hình với cấu hình quantization 4-bit

In [5]:
torch.cuda.empty_cache()
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
 
base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"
 
print(f"Loading tokenizer from {base_model_id}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")
 
print(f"Loading base model from {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)
base_model.config.pad_token_id = tokenizer.pad_token_id
print("Base model loaded in 4-bit")

Loading tokenizer from mistralai/Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Base model loaded in 4-bit


## 6. Load Fine-tuned Adapter & Merge

Load PEFT adapter đã được fine-tune từ stage 1

In [6]:
adapter_model_id = "/kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best"
if not os.path.exists(adapter_model_id):
    raise FileNotFoundError(f"Adapter path not found: {adapter_model_id}")
print(f"Found adapter at: {adapter_model_id}")
 
model = PeftModel.from_pretrained(base_model, adapter_model_id)
print("PEFT adapter loaded successfully")

Found adapter at: /kaggle/input/datasets/linhtron123/finetune-mistral/final_model_best
PEFT adapter loaded successfully


## 7. Load Test Data

Load dataset chứa:
- **Dialogue**: Cuộc hội thoại giữa plaintiff và defendant
- **PLAINTIFF intent**: Ý định của nguyên đơn
- **DEFENDANT intent**: Ý định của bị đơn

In [7]:
def load_csv(file_path):
    return pd.read_csv(file_path)
 
# Dataset WITH intent (used by ablation A & C)
csv_intent = load_csv("/kaggle/input/datasets/linhtron123/test-intent-p2/test_intent.csv")
dialogues_intent    = csv_intent["Dialogue"].tolist()
plaintiff_intents   = csv_intent["PLAINTIFF"].tolist()
defendant_intents   = csv_intent["DEFENDANT"].tolist()
 
# Dataset WITHOUT intent (used by ablation B)
csv_no_intent = load_csv("/kaggle/input/datasets/linhtron123/legaldata/test_split.csv")
dialogues_no_intent = csv_no_intent["Dialogue"].tolist()

## 8. Định nghĩa Manipulation Tactics

Danh sách 11 kỹ thuật manipulation được model nhận diện

In [8]:
ALLOWED_TACTICS = [
    "persuasion", "playing the victim", "gaslighting", "evasion",
    "deflection", "minimization", "dismissal", "guilt tripping",
    "emotional appeal", "framing the narrative", "character attack"
]

## 9. Hàm Generation

Hàm sinh text từ model

In [9]:
def generate_response(prompt, max_new_tokens=1000):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 8. Legal-BERT Classifier (used by ablation B) 

In [10]:
class ManipulationClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.35, k_layers=3):
        super().__init__()
        self.k_layers = k_layers
        self.encoder = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.LayerNorm(hidden_size // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_labels)
        )
 
    def max_cls_pool(self, hidden_states):
        cls_per_layer = torch.stack(
            [hidden_states[-(i + 1)][:, 0, :] for i in range(self.k_layers)], dim=1
        )
        return cls_per_layer.max(dim=1).values
 
    def forward(self, input_ids, attention_mask, labels=None):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.max_cls_pool(out.hidden_states)
        logits = self.classifier(pooled)
        return None, logits

In [11]:
class TechniqueClassifier:
    def __init__(self, model_path, optuna_path, device="cuda"):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
        with open(optuna_path, "r") as f:
            best_params = json.load(f)["best_params"]
        self.model = ManipulationClassifier(
            "nlpaueb/legal-bert-base-uncased",
            num_labels=11,
            dropout=best_params["dropout"],
            k_layers=3
        )
        self.model.load_state_dict(torch.load(f"{model_path}/best_model.pt", map_location=device))
        self.model.to(device)
        self.model.eval()
        self.thresholds = np.load(f"{model_path}/best_thresholds.npy")
        self.techniques_list = ALLOWED_TACTICS
        print(f"  [BERT] Model loaded | thresholds: {[round(t,2) for t in self.thresholds]}")
 
    def predict(self, dialogue):
        tokens = self.tokenizer.encode(dialogue, add_special_tokens=False)
        chunks_input_ids, chunks_attention_mask = [], []
        for start in range(0, max(len(tokens), 1), 256):
            end   = min(start + 512 - 2, len(tokens))
            chunk = [self.tokenizer.cls_token_id] + tokens[start:end] + [self.tokenizer.sep_token_id]
            pad_len        = 512 - len(chunk)
            attention_mask = [1] * len(chunk) + [0] * pad_len
            chunk          = chunk + [self.tokenizer.pad_token_id] * pad_len
            chunks_input_ids.append(chunk)
            chunks_attention_mask.append(attention_mask)
            if end == len(tokens):
                break
        input_ids      = torch.tensor(chunks_input_ids).to(self.device)
        attention_mask = torch.tensor(chunks_attention_mask).to(self.device)
        with torch.no_grad():
            _, logits = self.model(input_ids, attention_mask)
            probs     = torch.sigmoid(logits).cpu().numpy()
        final_probs = probs.max(axis=0)
        pred_techniques = [
            self.techniques_list[i] for i, p in enumerate(final_probs)
            if p >= self.thresholds[i]
        ]
        confidences = {self.techniques_list[i]: float(final_probs[i]) for i in range(len(final_probs))}
        return pred_techniques, confidences
 
 
BERT_CLASSIFIER = TechniqueClassifier(
    model_path="/kaggle/input/datasets/anhtuan23004/no-intent-legalbert/bert_final_no_intent",
    optuna_path="/kaggle/input/datasets/anhtuan23004/no-intent-legalbert/bert_final_no_intent/optuna_results.json"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

  [BERT] Model loaded | thresholds: [np.float64(0.53), np.float64(0.58), np.float64(0.58), np.float64(0.45), np.float64(0.5), np.float64(0.6), np.float64(0.5), np.float64(0.5), np.float64(0.48), np.float64(0.5), np.float64(0.4)]


In [12]:
INVALID_MANIPULATORS = {"None", "Unknown", "nan", "none", "unknown"}


# ABLATION A — No Multi-Agent  (single LLM call, with intent)

In [13]:
def analyze_dialogue_no_multiagent(dialogue, plaintiff_intent, defendant_intent):
    prompt = f"""<s>[INST] You are a manipulation detection expert analyzing a legal dialogue.
 
Defendant's Intent: {defendant_intent}
Plaintiff's Intent: {plaintiff_intent}
Dialogue: {dialogue}
 
### Task
1. Determine if manipulation is present in this dialogue.
2. If yes, identify who is the primary manipulator (Plaintiff or Defendant).
3. If yes, list the manipulation techniques used from this list ONLY:
   {', '.join(ALLOWED_TACTICS)}
 
### Output format (follow exactly)
Manipulation Present: <No or Yes>
Primary Manipulator: <Plaintiff or Defendant or None>
Techniques: <comma-separated techniques, or None>
[/INST]"""
 
    raw = generate_response(prompt, max_new_tokens=150)
    print(f"  [LLM raw] {raw[:200]!r}")
 
    manip_match          = re.search(r"manipulation present\s*:\s*(yes|no)", raw, re.IGNORECASE)
    manipulation_present = "Yes" if (manip_match and manip_match.group(1).lower() == "yes") else "No"
 
    primary_manipulator  = "None"
    manip_person_match   = re.search(r"primary manipulator\s*:\s*(plaintiff|defendant|none)", raw, re.IGNORECASE)
    if manip_person_match:
        val = manip_person_match.group(1).capitalize()
        primary_manipulator = val if val in ("Plaintiff", "Defendant") else "None"
 
    techniques = []
    tech_match = re.search(r"techniques\s*:\s*(.+)", raw, re.IGNORECASE)
    if tech_match:
        raw_techs = tech_match.group(1).strip()
        if raw_techs.lower() != "none":
            for token in raw_techs.split(","):
                token = token.strip().lower()
                for tactic in ALLOWED_TACTICS:
                    if token == tactic:
                        techniques.append(tactic)
                        break
 
    if manipulation_present == "No":
        primary_manipulator = "None"
        techniques = []
 
    return {
        "manipulation_present": manipulation_present,
        "primary_manipulator":  primary_manipulator,
        "techniques":           techniques,
    }

In [14]:
def run_ablation_a():
    manipulation_results, manipulator_results, techniques_results = [], [], []
 
    for idx in range(len(dialogues_intent)):
        print(f"\n=== [A] Dialogue {idx}/{len(dialogues_intent)-1} ===")
        try:
            result        = analyze_dialogue_no_multiagent(
                dialogue         = dialogues_intent[idx],
                plaintiff_intent = plaintiff_intents[idx],
                defendant_intent = defendant_intents[idx],
            )
            manip_present = result["manipulation_present"]
            manip_person  = result["primary_manipulator"]
            techs         = result["techniques"]
        except Exception:
            import traceback; traceback.print_exc()
            manip_present, manip_person, techs = "Error", "Error", []
 
        manipulation_results.append(manip_present)
        manipulator_results.append(manip_person)
        techniques_results.append(", ".join(techs) if techs else "None")
        print(f"  → {manip_present} | {manip_person} | {', '.join(techs) if techs else 'None'}")
 
        if idx % 20 == 0 and idx > 0:
            pd.DataFrame({
                "Dialogue_Index":       list(range(idx + 1)),
                "Dialogue":             dialogues_intent[:idx + 1],
                "Manipulation_Present": manipulation_results,
                "Primary_Manipulator":  manipulator_results,
                "Techniques_Used":      techniques_results,
            }).to_csv("/kaggle/working/checkpoint_ablation_a.csv", index=False)
            print(f"  [Checkpoint saved at idx={idx}]")
 
    output_df = csv_intent[["Dialogue", "PLAINTIFF", "DEFENDANT"]].copy()
    output_df.insert(0, "Dialogue_Index", range(len(output_df)))
    output_df["Manipulation_Present"] = manipulation_results
    output_df["Primary_Manipulator"]  = manipulator_results
    output_df["Techniques_Used"]      = techniques_results
 
    output_path = "/kaggle/working/ablation_llm_only_results.csv"
    output_df.to_csv(output_path, index=False)
    print(f"\n[A DONE] Results saved to: {output_path}")
    return output_path

# ABLATION B — No Intent  (multi-agent + BERT, no intent)

In [15]:
class AgentStateB(TypedDict):
    dialogue:              str
    dialogue_index:        int
    manipulation_detected: Optional[str]
    primary_manipulator:   Optional[str]
    evidence:              Optional[str]
    final_summary:         Optional[Dict[str, Any]]
    messages:              List[Union[HumanMessage, AIMessage]]
    techniques_llm:        Optional[List[str]]
    techniques_bert:       Optional[List[str]]
    techniques_used:       Optional[List[str]]
    bert_confidences:      Optional[Dict[str, float]]
    bert_failed:           Optional[bool]

In [16]:
def detector_agent_b(state):
    dialogue = state["dialogue"]
    system_prompt = f"""<s>[INST] You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.
### Instructions:
1. Read the dialogue carefully.
2. Determine if any manipulation is present or not.
 
Dialogue: {dialogue}
Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: No or Yes
[/INST]"""
 
    response           = generate_response(system_prompt, max_new_tokens=5)
    model_response     = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    extracted_response = model_response.group(1).strip() if model_response else response.strip()
 
    lines = [line.strip() for line in extracted_response.split('\n') if line.strip()]
    if lines:
        last_line = lines[-1]
        if last_line.lower() == "yes":
            result, status = "Yes", "Yes (exact match)"
        elif last_line.lower() == "no":
            result, status = "No", "No (exact match)"
        else:
            result, status = "No", f"No (default, got: '{last_line}')"
    else:
        result, status = "No", "No (default, empty response)"
 
    print(f"  [Detector-B] raw='{extracted_response.strip()}' → {status}")
    return {**state, "manipulation_detected": result,
            "messages": state["messages"] + [AIMessage(content=f"Manipulation Present: {result}")]}

In [17]:
def analyzer_agent_b(state):
    if state["manipulation_detected"] == "No":
        return {**state, "primary_manipulator": "None", "techniques_llm": [],
                "messages": state["messages"] + [AIMessage(content="No manipulation detected, skipping analysis.")]}
 
    print("Manipulation detected, analyzing manipulator and techniques...")
    combined_prompt = f"""<s>[INST] Compare both dialogue, and analyze the manipulation:
        Dialogue: {state['dialogue']}
        First, determine who is the primary manipulator (answer with ONLY Plaintiff or Defendant).
        Second, identify which manipulation techniques they used from this list:
        {', '.join(ALLOWED_TACTICS)}
        Format your response exactly like this:
        Primary Manipulator: [ ]
        Techniques: [list ONLY the techniques that were actually used, separated by commas]
   [/INST]"""
 
    combined_response = generate_response(combined_prompt, max_new_tokens=100)
 
    primary_manipulator = "Unknown"
    if "primary manipulator: plaintiff" in combined_response.lower():
        primary_manipulator = "Plaintiff"
    elif "primary manipulator: defendant" in combined_response.lower():
        primary_manipulator = "Defendant"
 
    techniques_llm = []
    for line in combined_response.split('\n'):
        if line.lower().startswith("techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            for technique in techniques_part.split(','):
                technique = technique.strip()
                for allowed_tactic in ALLOWED_TACTICS:
                    if technique.lower() == allowed_tactic.lower():
                        techniques_llm.append(allowed_tactic)
                        break
 
    print(f"  [Analyzer-B] Manipulator: {primary_manipulator} | Techniques(LLM): {techniques_llm or 'None'}")
    return {**state, "primary_manipulator": primary_manipulator, "techniques_llm": techniques_llm,
            "messages": state["messages"] + [AIMessage(content=f"Primary Manipulator: {primary_manipulator}\nTechniques: {techniques_llm}")]}

In [18]:
def bert_technique_agent_b(state):
    if state["manipulation_detected"] == "No":
        return {**state, "techniques_bert": [], "bert_confidences": {}, "bert_failed": False,
                "evidence": "No manipulation detected."}
 
    techniques, confidence = BERT_CLASSIFIER.predict(dialogue=state["dialogue"])
    max_conf    = max(confidence.values()) if confidence else 0
    bert_failed = max_conf < 0.05
 
    if bert_failed:
        print(f"  [BERT] FAILED (max_conf={max_conf:.4f}) → Meta Agent use results from LLM")
    else:
        print(f"  [BERT] {techniques if techniques else 'None'} (max_conf={max_conf:.3f})")
 
    evidence  = f"BERT predictions: {techniques}\n"
    evidence += f"BERT failed: {bert_failed}\n"
    evidence += "Confidence: " + str({k: round(v, 3) for k, v in confidence.items() if v > 0.2})
 
    return {**state, "techniques_bert": techniques, "bert_confidences": confidence,
            "bert_failed": bert_failed, "evidence": evidence,
            "messages": state["messages"] + [AIMessage(content=evidence)]}

In [19]:
def meta_agent_b(state):
    det_present     = state.get("manipulation_detected", "No")
    pri_manipulator = state.get("primary_manipulator", "None") or "None"
    bert_techs      = state.get("techniques_bert", []) or []
    llm_techs       = state.get("techniques_llm", []) or []
    bert_failed     = state.get("bert_failed", False)
 
    has_manipulation = (det_present == "Yes")
    has_manipulator  = pri_manipulator not in INVALID_MANIPULATORS
 
    if has_manipulation and has_manipulator:
        if not bert_failed and len(bert_techs) > 0:
            final_verdict, final_manipulator, final_tech_list = "Yes", pri_manipulator, bert_techs
            case_note = "BERT Direct Output"
        elif len(llm_techs) > 0:
            final_verdict, final_manipulator, final_tech_list = "Yes", pri_manipulator, llm_techs
            case_note = "LLM Fallback (BERT Failed)"
        else:
            final_verdict, final_manipulator, final_tech_list = "No", "None", []
            case_note = "No techniques found by both models"
    else:
        final_verdict, final_manipulator, final_tech_list = "No", "None", []
        case_note = "No manipulation detected"
 
    print(f"  [Meta-B] {case_note} | Final: {final_verdict} | {final_manipulator} | {final_tech_list}")
    return {**state, "techniques_used": final_tech_list,
            "final_summary": {"manipulation_present": final_verdict,
                              "primary_manipulator":  final_manipulator,
                              "techniques":           final_tech_list}}

In [20]:
workflow_b = StateGraph(AgentStateB)
workflow_b.add_node("detector_agent",       detector_agent_b)
workflow_b.add_node("analyzer_agent",       analyzer_agent_b)
workflow_b.add_node("bert_technique_agent", bert_technique_agent_b)
workflow_b.add_node("meta_agent",           meta_agent_b)
workflow_b.set_entry_point("detector_agent")
workflow_b.add_conditional_edges(
    "detector_agent",
    lambda state: "analyzer_agent" if state["manipulation_detected"] == "Yes" else "meta_agent"
)
workflow_b.add_edge("analyzer_agent",       "bert_technique_agent")
workflow_b.add_edge("bert_technique_agent", "meta_agent")
workflow_b.add_edge("meta_agent",           END)
workflow_compiled_b = workflow_b.compile()


In [21]:
def process_dialogue_b(dialogue_idx):
    return workflow_compiled_b.invoke(AgentStateB({
        "dialogue":              dialogues_no_intent[dialogue_idx],
        "dialogue_index":        dialogue_idx,
        "manipulation_detected": None,
        "primary_manipulator":   "None",
        "evidence":              None,
        "final_summary":         None,
        "bert_failed":           False,
        "crosscheck_mode":       None,
        "messages": [HumanMessage(content=f"Analyze dialogue {dialogue_idx}")],
        "techniques_llm":        [],
        "techniques_bert":       [],
        "techniques_used":       [],
        "bert_confidences":      {},
    }))
 

In [22]:
def run_ablation_b():
    manipulation_results, manipulator_results = [], []
    techniques_results, techniques_llm_results, techniques_bert_results = [], [], []
 
    for idx in range(len(dialogues_no_intent)):
        print(f"\n=== [B] Dialogue {idx}/{len(dialogues_no_intent)-1} ===")
        try:
            result              = process_dialogue_b(idx)
            summary             = result.get('final_summary', {})
            manip_present       = summary.get('manipulation_present', 'No')
            manip_person        = summary.get('primary_manipulator',  'None')
            techs               = summary.get('techniques', [])
            techniques_llm_val  = ", ".join(result.get("techniques_llm",  []) or []) or "None"
            techniques_bert_val = ", ".join(result.get("techniques_bert", []) or []) or "None"
        except Exception:
            import traceback; traceback.print_exc()
            manip_present = manip_person = 'Error'
            techs = []
            techniques_llm_val = techniques_bert_val = 'Error'
 
        manipulation_results.append(manip_present)
        manipulator_results.append(manip_person)
        techniques_results.append(', '.join(techs) if techs else 'None')
        techniques_llm_results.append(techniques_llm_val)
        techniques_bert_results.append(techniques_bert_val)
        print(f"  → {manip_present} | {manip_person} | {', '.join(techs) if techs else 'None'}")
 
        if idx % 20 == 0 and idx > 0:
            pd.DataFrame({
                'Dialogue_Index':       list(range(idx + 1)),
                'Dialogue':             dialogues_no_intent[:idx + 1],
                'Manipulation_Present': manipulation_results,
                'Primary_Manipulator':  manipulator_results,
                'Techniques_Used':      techniques_results,
                'Techniques_LLM':       techniques_llm_results,
                'Techniques_BERT':      techniques_bert_results,
            }).to_csv("/kaggle/working/checkpoint_ablation_b.csv", index=False)
            print(f"  [Checkpoint saved at idx={idx}]")
 
    output_df = csv_no_intent[['Dialogue']].copy()
    output_df.insert(0, 'Dialogue_Index', range(len(output_df)))
    output_df['Manipulation_Present'] = manipulation_results
    output_df['Primary_Manipulator']  = manipulator_results
    output_df['Techniques_Used']      = techniques_results
    output_df['Techniques_LLM']       = techniques_llm_results
    output_df['Techniques_BERT']      = techniques_bert_results
 
    output_path = "/kaggle/working/manipulation_results_no_intent.csv"
    output_df.to_csv(output_path, index=False)
    print(f"\n[B DONE] Results saved to: {output_path}")
    return output_path

# ABLATION C — No Legal-BERT  (multi-agent + intent, LLM only)

In [23]:
 class AgentStateC(TypedDict):
    dialogue:              str
    dialogue_index:        int
    plaintiff_intent:      str
    defendant_intent:      str
    manipulation_detected: Optional[str]
    primary_manipulator:   Optional[str]
    evidence:              Optional[str]
    final_summary:         Optional[Dict[str, Any]]
    messages:              List[Union[HumanMessage, AIMessage]]
    techniques_llm:        Optional[List[str]]
    techniques_used:       Optional[List[str]]

In [24]:
def detector_agent_c(state):
    dialogue         = state["dialogue"]
    plaintiff_intent = state["plaintiff_intent"]
    defendant_intent = state["defendant_intent"]
 
    system_prompt = f"""<s>[INST] You are a manipulation detection expert. Your task is to analyze this dialogue and determine if there is manipulation present.
### Instructions:
1. Read the dialogue carefully.
2. Analyze it with the context of the intents provided
3. Determine if any manipulation is present or not.
Defendant's Intent: {defendant_intent}
Plaintiff's Intent: {plaintiff_intent}
Dialogue: {dialogue}
Question: Is there manipulation present in this dialogue?
You must respond with ONLY one word: Yes or No
[/INST]"""
 
    response           = generate_response(system_prompt, max_new_tokens=5)
    model_response     = re.search(r"\[/INST\](.*?)$", response, re.DOTALL)
    extracted_response = model_response.group(1).strip() if model_response else response.strip()
 
    lines = [line.strip() for line in extracted_response.split('\n') if line.strip()]
    if lines:
        last_line = lines[-1]
        if last_line.lower() == "yes":
            result, status = "Yes", "Yes (exact match)"
        elif last_line.lower() == "no":
            result, status = "No", "No (exact match)"
        else:
            result, status = "No", f"No (default, got: '{last_line}')"
    else:
        result, status = "No", "No (default, empty response)"
 
    print(f"  [Detector-C] raw='{extracted_response.strip()}' → {status}")
    return {**state, "manipulation_detected": result,
            "messages": state["messages"] + [AIMessage(content=f"Manipulation Present: {result}")]}


In [25]:
def analyzer_agent_c(state):
    if state["manipulation_detected"] == "No":
        return {**state, "primary_manipulator": "None", "techniques_llm": [],
                "messages": state["messages"] + [AIMessage(content="No manipulation detected, skipping analysis.")]}
 
    print("Manipulation detected, analyzing manipulator and techniques...")
    combined_prompt = f"""<s>[INST] Compare both dialogue and the given intents of both defendant and plaintiff, and analyze the manipulation:
        Defendant's Intent: {state['defendant_intent']}
        Plaintiff's Intent: {state['plaintiff_intent']}
        Dialogue: {state['dialogue']}
        First, determine who is the primary manipulator (answer with ONLY Defendant or Plaintiff).
        Second, identify which manipulation techniques they used from this list:
        {', '.join(ALLOWED_TACTICS)}
        Format your response exactly like this:
        Primary Manipulator: [ ]
        Techniques: [list ONLY the techniques that were actually used, separated by commas]
   [/INST]"""
 
    combined_response   = generate_response(combined_prompt, max_new_tokens=100)
    primary_manipulator = "Unknown"
    if "primary manipulator: plaintiff" in combined_response.lower():
        primary_manipulator = "Plaintiff"
    elif "primary manipulator: defendant" in combined_response.lower():
        primary_manipulator = "Defendant"
 
    techniques_llm = []
    for line in combined_response.split('\n'):
        if line.lower().startswith("techniques:"):
            techniques_part = line.split(':', 1)[1].strip()
            for technique in techniques_part.split(','):
                technique = technique.strip()
                for allowed_tactic in ALLOWED_TACTICS:
                    if technique.lower() == allowed_tactic.lower():
                        techniques_llm.append(allowed_tactic)
                        break
 
    print(f"  [Analyzer-C] Manipulator: {primary_manipulator} | Techniques(LLM): {techniques_llm or 'None'}")
    return {**state, "primary_manipulator": primary_manipulator, "techniques_llm": techniques_llm,
            "messages": state["messages"] + [AIMessage(content=f"Primary Manipulator: {primary_manipulator}\nTechniques: {techniques_llm}")]}

In [26]:
def meta_agent_c(state):
    det_present     = state.get("manipulation_detected", "No")
    pri_manipulator = state.get("primary_manipulator", "None") or "None"
    llm_techs       = state.get("techniques_llm", []) or []
 
    has_manipulation = (det_present == "Yes")
    has_manipulator  = pri_manipulator not in INVALID_MANIPULATORS
 
    if has_manipulation and has_manipulator:
        final_verdict, final_manipulator, final_tech_list = "Yes", pri_manipulator, llm_techs
        case_note = "Full signal (LLM only)"
    else:
        final_verdict, final_manipulator, final_tech_list = "No", "None", []
        case_note = "No manipulation or invalid manipulator"
 
    print(f"  [Meta-C] {case_note} | Final: {final_verdict} | {final_manipulator} | {final_tech_list}")
    return {**state, "techniques_used": final_tech_list,
            "final_summary": {"manipulation_present": final_verdict,
                              "primary_manipulator":  final_manipulator,
                              "techniques":           final_tech_list}}

In [27]:
workflow_c = StateGraph(AgentStateC)
workflow_c.add_node("detector_agent", detector_agent_c)
workflow_c.add_node("analyzer_agent", analyzer_agent_c)
workflow_c.add_node("meta_agent",     meta_agent_c)
workflow_c.set_entry_point("detector_agent")
workflow_c.add_conditional_edges(
    "detector_agent",
    lambda state: "analyzer_agent" if state["manipulation_detected"] == "Yes" else "meta_agent"
)
workflow_c.add_edge("analyzer_agent", "meta_agent")
workflow_c.add_edge("meta_agent",     END)
workflow_compiled_c = workflow_c.compile()

In [28]:
def process_dialogue_c(dialogue_idx):
    return workflow_compiled_c.invoke(AgentStateC({
        "dialogue":              dialogues_intent[dialogue_idx],
        "dialogue_index":        dialogue_idx,
        "plaintiff_intent":      plaintiff_intents[dialogue_idx],
        "defendant_intent":      defendant_intents[dialogue_idx],
        "manipulation_detected": None,
        "primary_manipulator":   "None",
        "evidence":              None,
        "final_summary":         None,
        "messages": [HumanMessage(content=f"Analyze dialogue {dialogue_idx}")],
        "techniques_llm":        [],
        "techniques_used":       [],
    }))

In [29]:
def run_ablation_c():
    manipulation_results, manipulator_results = [], []
    techniques_results, techniques_llm_results = [], []
 
    for idx in range(len(dialogues_intent)):
        print(f"\n=== [C] Dialogue {idx}/{len(dialogues_intent)-1} ===")
        try:
            result             = process_dialogue_c(idx)
            summary            = result.get('final_summary', {})
            manip_present      = summary.get('manipulation_present', 'No')
            manip_person       = summary.get('primary_manipulator',  'None')
            techs              = summary.get('techniques', [])
            techniques_llm_val = ", ".join(result.get("techniques_llm", []) or []) or "None"
        except Exception:
            import traceback; traceback.print_exc()
            manip_present = manip_person = 'Error'
            techs = []
            techniques_llm_val = 'Error'
 
        manipulation_results.append(manip_present)
        manipulator_results.append(manip_person)
        techniques_results.append(', '.join(techs) if techs else 'None')
        techniques_llm_results.append(techniques_llm_val)
        print(f"  → {manip_present} | {manip_person} | {', '.join(techs) if techs else 'None'}")
 
        if idx % 20 == 0 and idx > 0:
            pd.DataFrame({
                'Dialogue_Index':       list(range(idx + 1)),
                'Dialogue':             dialogues_intent[:idx + 1],
                'Manipulation_Present': manipulation_results,
                'Primary_Manipulator':  manipulator_results,
                'Techniques_Used':      techniques_results,
                'Techniques_LLM':       techniques_llm_results,
            }).to_csv("/kaggle/working/checkpoint_ablation_c.csv", index=False)
            print(f"  [Checkpoint saved at idx={idx}]")
 
    output_df = csv_intent[['Dialogue', 'PLAINTIFF', 'DEFENDANT']].copy()
    output_df.insert(0, 'Dialogue_Index', range(len(output_df)))
    output_df['Manipulation_Present'] = manipulation_results
    output_df['Primary_Manipulator']  = manipulator_results
    output_df['Techniques_Used']      = techniques_results
    output_df['Techniques_LLM']       = techniques_llm_results
 
    output_path = "/kaggle/working/ablation_no_bert_results.csv"
    output_df.to_csv(output_path, index=False)
    print(f"\n[C DONE] Results saved to: {output_path}")
    return output_path

In [30]:
print("\n" + "=" * 60)
print("Running Ablation A: No Multi-Agent")
run_ablation_a()


Running Ablation A: No Multi-Agent

=== [A] Dialogue 0/154 ===
  [LLM raw] 'Manipulation Present: Yes\nPrimary Manipulator: Defendant\nTechniques: evasion, deflection, minimization'
  → Yes | Defendant | evasion, deflection, minimization

=== [A] Dialogue 1/154 ===
  [LLM raw] 'Manipulation Present: Yes\nPrimary Manipulator: Defendant\nTechniques: playing the victim, evasion, deflection'
  → Yes | Defendant | playing the victim, evasion, deflection

=== [A] Dialogue 2/154 ===
  [LLM raw] 'Manipulation Present: Yes\nPrimary Manipulator: Plaintiff\nTechniques: emotional appeal, framing the narrative'
  → Yes | Plaintiff | emotional appeal, framing the narrative

=== [A] Dialogue 3/154 ===
  [LLM raw] 'Manipulation Present: Yes\nPrimary Manipulator: Defendant\nTechniques: playing the victim, deflection, emotional appeal'
  → Yes | Defendant | playing the victim, deflection, emotional appeal

=== [A] Dialogue 4/154 ===
  [LLM raw] 'Manipulation Present: No\nPrimary Manipulator: None\nTech

'/kaggle/working/ablation_llm_only_results.csv'

In [31]:
print("\n" + "=" * 60)
print("Running Ablation B: No Intent")
run_ablation_b()


Running Ablation B: No Intent

=== [B] Dialogue 0/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 1/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 2/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 3/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 4/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 5/154 ===
  [Detector-B] raw='Yes' → Yes (exact match)
Manipulation detected, analyzing manipulator and techniques...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1196 > 512). Running this sequence through the model will result in indexing errors


  [Analyzer-B] Manipulator: Plaintiff | Techniques(LLM): None
  [BERT] ['persuasion', 'playing the victim', 'gaslighting', 'evasion', 'deflection', 'emotional appeal', 'framing the narrative'] (max_conf=0.664)
  [Meta-B] BERT Direct Output | Final: Yes | Plaintiff | ['persuasion', 'playing the victim', 'gaslighting', 'evasion', 'deflection', 'emotional appeal', 'framing the narrative']
  → Yes | Plaintiff | persuasion, playing the victim, gaslighting, evasion, deflection, emotional appeal, framing the narrative

=== [B] Dialogue 6/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 7/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 8/154 ===
  [Detector-B] raw='No' → No (exact match)
  [Meta-B] No manipulation detected | Final: No | None | []
  → No | None | None

=== [B] Dialogue 9/154 ===

'/kaggle/working/manipulation_results_no_intent.csv'

In [32]:
print("\n" + "=" * 60)
print("Running Ablation C: No Legal-BERT")
run_ablation_c()


Running Ablation C: No Legal-BERT

=== [C] Dialogue 0/154 ===
  [Detector-C] raw='No' → No (exact match)
  [Meta-C] No manipulation or invalid manipulator | Final: No | None | []
  → No | None | None

=== [C] Dialogue 1/154 ===
  [Detector-C] raw='Yes' → Yes (exact match)
Manipulation detected, analyzing manipulator and techniques...
  [Analyzer-C] Manipulator: Defendant | Techniques(LLM): None
  [Meta-C] Full signal (LLM only) | Final: Yes | Defendant | []
  → Yes | Defendant | None

=== [C] Dialogue 2/154 ===
  [Detector-C] raw='Yes' → Yes (exact match)
Manipulation detected, analyzing manipulator and techniques...
  [Analyzer-C] Manipulator: Plaintiff | Techniques(LLM): ['emotional appeal', 'framing the narrative']
  [Meta-C] Full signal (LLM only) | Final: Yes | Plaintiff | ['emotional appeal', 'framing the narrative']
  → Yes | Plaintiff | emotional appeal, framing the narrative

=== [C] Dialogue 3/154 ===
  [Detector-C] raw='Yes' → Yes (exact match)
Manipulation detected, analyz

'/kaggle/working/ablation_no_bert_results.csv'

# EVALUATION FOR ABLATION A, B, C

In [33]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, jaccard_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MultiLabelBinarizer

def parse_techniques(x):
    if pd.isna(x) or str(x).strip().lower() in ["none", "nan", ""]:
        return ["none"]
    return [t.strip().lower() for t in str(x).split(",") if t.strip()]

def evaluate_ablation(pred_path, ablation_name):
    print("\n" + "=" * 60)
    print(f"  EVALUATION: {ablation_name}")
    print("=" * 60)

    df_pred = pd.read_csv(pred_path)
    df_test = pd.read_csv('/kaggle/input/datasets/linhtron123/legaldata/test_split.csv')

    test = pd.merge(df_pred, df_test, on='Dialogue', how='left')
    print(f"Dataset size after merge: {len(test)}")

    # ----------------------------------------------------------
    # TASK 1: Manipulation Detection (Binary)
    # ----------------------------------------------------------
    test["Manipulation_Present"] = test["Manipulation_Present"].map({'Yes': 1, 'No': 0})
    targets_bin  = test["Manipulative"].astype(int).tolist()
    preds_bin    = test["Manipulation_Present"].astype(int).tolist()

    acc_t1  = accuracy_score(targets_bin, preds_bin)
    prec_t1 = precision_score(targets_bin, preds_bin, zero_division=0)
    rec_t1  = recall_score(targets_bin, preds_bin, zero_division=0)
    f1_t1   = f1_score(targets_bin, preds_bin, average="binary", zero_division=0)
    cm_t1   = confusion_matrix(targets_bin, preds_bin)

    print("\n--- TASK 1: Manipulation Detection (Binary) ---")
    print(f"Accuracy:  {acc_t1:.4f}")
    print(f"Precision: {prec_t1:.4f}")
    print(f"Recall:    {rec_t1:.4f}")
    print(f"F1 Score:  {f1_t1:.4f}")
    print(f"Confusion Matrix:")
    print(f"              Predicted")
    print(f"            No      Yes")
    print(f"Actual No  {cm_t1[0][0]:4d}    {cm_t1[0][1]:4d}")
    print(f"Actual Yes {cm_t1[1][0]:4d}    {cm_t1[1][1]:4d}")

    # ----------------------------------------------------------
    # TASK 2: Primary Manipulator Identification
    # ----------------------------------------------------------
    if "Primary_Manipulator" in test.columns:
        test["Primary_Manipulator"]  = test["Primary_Manipulator"].fillna("None").str.strip()
        test["Primary Manipulator"]  = test["Primary Manipulator"].fillna("None").str.strip()

        all_cats = list(set(
            test["Primary_Manipulator"].tolist() +
            test["Primary Manipulator"].tolist()
        ))
        enc = LabelEncoder()
        enc.fit(all_cats)

        tgt_enc  = enc.transform(test["Primary_Manipulator"])
        pred_enc = enc.transform(test["Primary Manipulator"])

        acc_t2  = accuracy_score(tgt_enc, pred_enc)
        prec_t2 = precision_score(tgt_enc, pred_enc, average="weighted", zero_division=0)
        rec_t2  = recall_score(tgt_enc, pred_enc, average="weighted", zero_division=0)
        f1_t2   = f1_score(tgt_enc, pred_enc, average="weighted", zero_division=0)
        cm_t2   = confusion_matrix(tgt_enc, pred_enc)

        print("\n--- TASK 2: Primary Manipulator Identification ---")
        print(f"Accuracy:  {acc_t2:.4f}")
        print(f"Precision: {prec_t2:.4f} (weighted)")
        print(f"Recall:    {rec_t2:.4f} (weighted)")
        print(f"F1 Score:  {f1_t2:.4f} (weighted)")
        print(f"Confusion Matrix:\n{cm_t2}")
        print(f"Label mapping: {dict(zip(enc.transform(enc.classes_), enc.classes_))}")
    else:
        print("\n--- TASK 2: Skipped (no Primary_Manipulator column) ---")
        acc_t2 = prec_t2 = rec_t2 = f1_t2 = None

    # ----------------------------------------------------------
    # TASK 3: Techniques Classification (Multi-label)
    # ----------------------------------------------------------
    test["Manipulation Techniques"] = test["Manipulation Techniques"].apply(parse_techniques)
    test["Techniques_Used"]         = test["Techniques_Used"].apply(parse_techniques)

    unique_gt   = set(l for labels in test["Manipulation Techniques"] for l in labels)
    unique_pred = set(l for labels in test["Techniques_Used"]         for l in labels)

    mlb = MultiLabelBinarizer()
    mlb.fit([list(unique_gt | unique_pred)])

    y_true = mlb.transform(test["Manipulation Techniques"])
    y_pred = mlb.transform(test["Techniques_Used"])

    acc_t3     = accuracy_score(y_true, y_pred)
    prec_t3    = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_t3     = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_t3      = f1_score(y_true, y_pred, average="macro", zero_division=0)
    jaccard_t3 = jaccard_score(y_true, y_pred, average="samples", zero_division=0)

    print("\n--- TASK 3: Techniques Classification (Multi-label) ---")
    print(f"Accuracy:      {acc_t3:.4f} (exact match)")
    print(f"Precision:     {prec_t3:.4f} (macro)")
    print(f"Recall:        {rec_t3:.4f} (macro)")
    print(f"F1 Score:      {f1_t3:.4f} (macro)")
    print(f"Jaccard Score: {jaccard_t3:.4f} (samples)")

    # ----------------------------------------------------------
    # Final Summary
    # ----------------------------------------------------------
    print("\n" + "-" * 60)
    print(f" FINAL SUMMARY — {ablation_name}")
    print("-" * 60)
    print(f"{'Task':<45} {'Metric':<12} {'Score'}")
    print("-" * 60)
    print(f"{'1. Manipulation Detection':<45} {'Accuracy':<12} {acc_t1:.4f}")
    print(f"{'':<45} {'F1':<12} {f1_t1:.4f}")
    if acc_t2 is not None:
        print(f"{'2. Primary Manipulator':<45} {'Accuracy':<12} {acc_t2:.4f}")
        print(f"{'':<45} {'F1 (wtd)':<12} {f1_t2:.4f}")
    print(f"{'3. Techniques Classification':<45} {'Accuracy':<12} {acc_t3:.4f}")
    print(f"{'':<45} {'F1 (macro)':<12} {f1_t3:.4f}")
    print(f"{'':<45} {'Jaccard':<12} {jaccard_t3:.4f}")
    print("=" * 60)


## Ablation A — No Multi-Agent

In [34]:
evaluate_ablation(
    pred_path          = "/kaggle/working/ablation_llm_only_results.csv",
    ablation_name      = "Ablation A — No Multi-Agent (LLM Only)",
)


  EVALUATION: Ablation A — No Multi-Agent (LLM Only)
Dataset size after merge: 155

--- TASK 1: Manipulation Detection (Binary) ---
Accuracy:  0.6774
Precision: 0.6552
Recall:    1.0000
F1 Score:  0.7917
Confusion Matrix:
              Predicted
            No      Yes
Actual No    10      50
Actual Yes    0      95

--- TASK 2: Primary Manipulator Identification ---
Accuracy:  0.4516
Precision: 0.6597 (weighted)
Recall:    0.4516 (weighted)
F1 Score:  0.4856 (weighted)
Confusion Matrix:
[[33 30  5]
 [ 0 10  0]
 [30 20 27]]
Label mapping: {np.int64(0): np.str_('Defendant'), np.int64(1): np.str_('None'), np.int64(2): np.str_('Plaintiff')}

--- TASK 3: Techniques Classification (Multi-label) ---
Accuracy:      0.0710 (exact match)
Precision:     0.1948 (macro)
Recall:        0.2691 (macro)
F1 Score:      0.1654 (macro)
Jaccard Score: 0.1949 (samples)

------------------------------------------------------------
 FINAL SUMMARY — Ablation A — No Multi-Agent (LLM Only)
--------------------

## Ablation B — No Intent


In [35]:
evaluate_ablation(
    pred_path          = "/kaggle/working/manipulation_results_no_intent.csv",
    ablation_name      = "Ablation B — No Intent (Multi-Agent + BERT)",
)


  EVALUATION: Ablation B — No Intent (Multi-Agent + BERT)
Dataset size after merge: 155

--- TASK 1: Manipulation Detection (Binary) ---
Accuracy:  0.7161
Precision: 1.0000
Recall:    0.5368
F1 Score:  0.6986
Confusion Matrix:
              Predicted
            No      Yes
Actual No    60       0
Actual Yes   44      51

--- TASK 2: Primary Manipulator Identification ---
Accuracy:  0.5742
Precision: 0.7828 (weighted)
Recall:    0.5742 (weighted)
F1 Score:  0.6173 (weighted)
Confusion Matrix:
[[17  0  0]
 [24 60 20]
 [22  0 12]]
Label mapping: {np.int64(0): np.str_('Defendant'), np.int64(1): np.str_('None'), np.int64(2): np.str_('Plaintiff')}

--- TASK 3: Techniques Classification (Multi-label) ---
Accuracy:      0.3871 (exact match)
Precision:     0.2773 (macro)
Recall:        0.4142 (macro)
F1 Score:      0.3172 (macro)
Jaccard Score: 0.4810 (samples)

------------------------------------------------------------
 FINAL SUMMARY — Ablation B — No Intent (Multi-Agent + BERT)
----------

## Ablation C — No Legal-BERT 

In [36]:
evaluate_ablation(
    pred_path          = "/kaggle/working/ablation_no_bert_results.csv",
    ablation_name      = "Ablation C — No Legal-BERT (Multi-Agent, LLM Only)",
)


  EVALUATION: Ablation C — No Legal-BERT (Multi-Agent, LLM Only)
Dataset size after merge: 155

--- TASK 1: Manipulation Detection (Binary) ---
Accuracy:  0.7806
Precision: 0.7402
Recall:    0.9895
F1 Score:  0.8468
Confusion Matrix:
              Predicted
            No      Yes
Actual No    27      33
Actual Yes    1      94

--- TASK 2: Primary Manipulator Identification ---
Accuracy:  0.6387
Precision: 0.7013 (weighted)
Recall:    0.6387 (weighted)
F1 Score:  0.6361 (weighted)
Confusion Matrix:
[[48 16  7]
 [ 0 27  1]
 [15 17 24]]
Label mapping: {np.int64(0): np.str_('Defendant'), np.int64(1): np.str_('None'), np.int64(2): np.str_('Plaintiff')}

--- TASK 3: Techniques Classification (Multi-label) ---
Accuracy:      0.2581 (exact match)
Precision:     0.1585 (macro)
Recall:        0.2453 (macro)
F1 Score:      0.1762 (macro)
Jaccard Score: 0.3380 (samples)

------------------------------------------------------------
 FINAL SUMMARY — Ablation C — No Legal-BERT (Multi-Agent, LLM On